# 01 - Annotate Controls

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir
sc.settings.verbosity = 2  # Show progress
sc.settings.set_figure_params(
    dpi=150,  # High-res output
    dpi_save=300,  # High-res when saving
    format="pdf",  # or 'pdf', 'svg'
    facecolor="white",  # or 'none' for transparent
    frameon=False,  # No outer frame
    vector_friendly=True,  # No rasterization warnings
    fontsize=10,  # Base font size
    figsize=(4, 4),  # Default figure size
    transparent=True,  # Transparent background if saving
)

mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_controls_processed.h5ad")

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
full_annotations = {
    # split early EEC progenitors
    "7_0": "Early EEC Progenitors",
    "7_1": "Early EEC Progenitors",
    "7_2": "Early EEC Progenitors",
    "7_3": "Paneth Cells",

    # goblet and TA clusters
    "0":    "Goblet Cells",
    "16":   "EC Cells",
    
    "22":   "Cycling TA Cells",
    "23":   "Cycling TA Cells",
    "24":   "Cycling TA Cells",
    
    "19":   "X Cells",
    "21":   "Stem Cells",
    "18":   "Late EEC Progenitors",

    # enterocytes
    "26":   "Enterocytes",
    "27":   "Enterocytes",
    "29":   "Enterocytes",

    # split endocrine subclusters
    "15_0": "I/N Cells",
    "15_1": "I/N Cells",
    "15_2": "K Cells",
    "15_3": "D Cells",
}


In [ ]:
adata_annotated = adata.copy()
adata_annotated.obs["cell_type"] = (
    adata_annotated.obs["clust"]
    .map(full_annotations)
    .fillna("TA Cells")
    .astype("category")
)

In [ ]:
sc.pl.embedding(adata_annotated, basis="X_umap", color=["sample", "clust", "cell_type"], wspace=0.4, frameon=False, legend_loc="on data", show=True)

In [ ]:
sc.pl.embedding(adata_annotated, basis="X_umap", color="cell_type", size=1, frameon=False, show=True)

In [ ]:
marker_dict = {
    "Stem Cells": ["LGR5", "SMOC2"],
    "TA Cells": ["OLFM4", "TLN2"],
    "Cycling TA Cells": ["MKI67", "TOP2A"],
    "Enterocytes": ["FABP1", "APOA4"],
    "BEST4+ Enterocytes": ["BEST4", "NOTCH2"],
    "Goblet Cells": ["FCGBP", "MUC2"],
    "Paneth Cells": ["DEFA5", "DEFA6"],
    "Tuft Cells": ["AVIL", "KIT"],
    "Early EEC Progenitors": ["NEUROG3", "ADGRL2"],
    "Late EEC Progenitors": ["RFX3"],
    "EC Cells": ["TPH1", "CHGB"],
    "D Cells": ["SST", "HHEX"],
    "X Cells": ["GHRL", "MLN"],
    "K Cells": ["GIP", "PLAGL1"],
    "I/N Cells": ["CCK", "NTS", "ONECUT3"],
    "L Cells": ["GCG", "PYY"],
}

In [ ]:
adata_annotated.obs["cell_type"] = adata_annotated.obs["cell_type"].astype("category").cat.reorder_categories([elem for elem in list(marker_dict.keys()) if elem in adata_annotated.obs["cell_type"].unique()])

In [ ]:
adata_annotated = adata_annotated[adata_annotated.obs["cell_type"].sort_values().index]

In [ ]:
sc.pl.dotplot(adata_annotated, var_names=marker_dict, groupby="cell_type", standard_scale="var", cmap="Reds", figsize=(12,5), use_raw=False, show=False)

In [ ]:
sc.pl.umap(adata_annotated, color=["BEST4", "AVIL"], use_raw=False, wspace=0.4, frameon=False, legend_loc="on data", show=True)

In [ ]:
adata_annotated.write(anndata_dir / "tf_ko_panel_combined_controls_annotated.h5ad")

## Measure hormonal levels

In [ ]:
adata_annotated = sc.read(anndata_dir / "tf_ko_panel_combined_controls_annotated.h5ad")

In [ ]:
adata_annotated.obs

In [ ]:
sc.pl.violin(adata_annotated, keys=["SST", "GIP", "GCG", "PYY", "CCK", "NTS", "GHRL", "TPH1", "GAST", "MLN", "PPY"], groupby="cell_type", use_raw=False, rotation=90)

## Plot figures

In [ ]:
adata_annotated = sc.read(anndata_dir / "tf_ko_panel_combined_controls_annotated.h5ad")

In [ ]:
# Subset to only the EEC lineage for better visualization of markers
adata_eecs = adata_annotated[adata_annotated.obs["cell_type"].isin(["Early EEC Progenitors", "Late EEC Progenitors", "EC Cells", "D Cells", "X Cells", "K Cells", "I/N Cells"])]

In [ ]:
adata_eecs.obs

In [ ]:
sc.pl.violin(adata_eecs, keys=["n_genes"], groupby="sample", use_raw=False, rotation=90)

In [ ]:
# Create highlight column
adata_eecs.obs["highlight"] = adata_eecs.obs["sample"].apply(
    lambda x: "Cnt4_2" if x == "Cnt4_2" else "Other"
)

adata_eecs.obs["highlight"] = pd.Categorical(
    adata_eecs.obs["highlight"],
    categories=["Other", "Cnt4_2"],
    ordered=True
)

# Sort so Cnt4_2 is plotted on top
adata_plot = adata_eecs[
    adata_eecs.obs["highlight"].sort_values().index
].copy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

sc.pl.umap(
    adata_plot,
    color="highlight",
    palette={"Other": "lightgray", "Cnt4_2": "black"},
    frameon=False,
    legend_loc="none",
    ax=axes[0],
    show=False,
    title="Cnt4_2"
)

sc.pl.umap(
    adata_plot,
    color="cell_type",
    frameon=False,
    ax=axes[1],
    show=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

In [ ]:
# Count cells per cell type and sample
counts = (
    adata_eecs.obs
    .groupby(["sample", "cell_type"])
    .size()
    .unstack(fill_value=0)
)

with plt.rc_context({"axes.grid": False}):
    # Plot stacked barplot
    ax = counts.plot(
        kind="bar",
        stacked=True,
        figsize=(5, 3),
        width=0.8
    )

    ax.set_ylabel("Total cell number")
    ax.set_xlabel("Sample")
    ax.legend(title="Cell type", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
# Subset to sample Cnt4_2 and plot GIP expression
adata_cnt4_2 = adata_eecs[adata_eecs.obs["sample"] == "Cnt4_2"]

sc.pl.umap(
    adata_cnt4_2,
    color="GIP",
    frameon=False,
    legend_loc="none",
    show=True,
    title="Cnt4_2 - GIP expression",
    use_raw=False
)

In [ ]:
sc.pl.dotplot(adata_annotated, var_names=marker_dict, groupby="cell_type", standard_scale="var", 
              cmap="Reds", figsize=(12,5), use_raw=False, show=False, save="_tf_ko_panel_combined_controls_annotated_dotplot.pdf")

In [ ]:
base_ct_colors = {}
for section, values in settings["ct_palette"].items():
    base_ct_colors.update(values)

In [ ]:
sc.pl.embedding(adata_annotated, basis="X_umap", color="cell_type", palette=base_ct_colors, frameon=False, show=True, save="_tf_ko_panel_combined_controls_annotated_umap.pdf")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scanpy as sc

# Base colormap
base_cmap = plt.cm.GnBu

# Create a new colormap with grey for zero
colors = [(0.7, 0.7, 0.7)] + [base_cmap(i) for i in range(base_cmap.N)]
new_cmap = mcolors.LinearSegmentedColormap.from_list("grey_GnBu", colors, N=base_cmap.N + 1)

In [ ]:
sc.pl.embedding(adata_annotated, 
                basis="X_umap", 
                color=["CHGA"], 
                frameon=False,
                show=True,
                size=5,
                color_map=new_cmap,
                use_raw=False)

In [ ]:
sc.pl.embedding(adata_annotated, 
                basis="X_umap", 
                color=["LGR5", "MKI67", "APOA4", "FCGBP", "DEFA5", 
                       "NEUROG3", "GHRL", "SST", "GIP", "CCK", "TPH1", "CHGA"], 
                frameon=False,
                show=True,
                size=5,
                color_map=new_cmap,
                use_raw=False,
                save="_tf_ko_panel_combined_controls_annotated_feature_exp.pdf")